In [31]:
#!pip install openai

In [7]:

import os, openai
from openai import AzureOpenAI
from dotenv import load_dotenv


#load_dotenv("/content/.env")
load_dotenv()

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")

In [8]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI.

    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.

    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.model_dump()  # Return the full response as dict

    except Exception as e:
        return {"error": str(e)}


In [34]:
#Define Your Prompts__
#Provide a series of prompts that guide the model through a chain of thought.
#Call the __get_completion__ to get a response from the AI model.
#Print both the prompt and the AI-generated response.

In [12]:
prompts = [
    "Imagine you are a detective trying to solve a mystery.",
    "You arrive at the crime scene and start looking for clues.",
    "You find a strange object at the crime scene. What is it?",
    "How does this object relate to the crime?",
    "Who do you think is the suspect and why?"
]


In [13]:
for prompt in prompts:
    response = get_completion(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()

Prompt: Imagine you are a detective trying to solve a mystery.
Response: {'id': 'chatcmpl-DSDFvjuH7k8SVJbmkku5iSOJDQAwq', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'Absolutely! I slip on my trench coat, adjust my fedora, and peer through the misty glow of the streetlamp. The city is quiet, but I know secrets lurk in the shadows. \n\n**Case File:** The Mystery of the Missing Manuscript\n\n**Background:**  \nLast night, the prized manuscript from the city’s library vanished without a trace. The only clues: a muddy footprint, a broken fountain pen, and a cryptic note that reads, “The truth is in the margins.”\n\n**Suspects:**  \n1. **Ms. Penelope Reed** – The meticulous librarian, last seen locking up.\n2. **Mr. Victor Slate** – The rival author, jealous of the manuscript’s fame.\n3. **Dr. Felix Marrow** – The historian, obsessed with rare texts.\n\n**My Approach:**  \nI’ll start by examining the clues. The muddy footprint—perhaps from the 

In [15]:
#As seen above: Each call to get_completion(prompt) is stateless.
'''
So the model:
--Does NOT remember previous prompts
--Treats every line like a brand new conversation

That’s why:
It asks for context again
It forgets the “object”
It can’t identify the suspect

'''

'\nSo the model:\n--Does NOT remember previous prompts\n--Treats every line like a brand new conversation\n\nThat’s why:\nIt asks for context again\nIt forgets the “object”\nIt can’t identify the suspect\n\n'

In [16]:
#First lets test and return only content
def get_completion(prompt, deployment_name=deployment_name):
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.choices[0].message.content  

    except Exception as e:
        return f"Error: {str(e)}"

In [17]:
for prompt in prompts:
    response = get_completion(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()

Prompt: Imagine you are a detective trying to solve a mystery.
Response: Absolutely, Detective [Your Name] reporting for duty! 🕵️‍♂️

**Case File:** The Curious Case of the Missing Manuscript

**Background:**  
Last night, at the prestigious Blackwood Manor, a priceless manuscript vanished from the library during a grand gala. The only clues left behind: a muddy footprint, a broken quill, and a cryptic note reading, "The truth hides in plain sight."

**Suspects:**  
1. **Lady Evelyn Blackwood** – The manor’s owner, known for her sharp wit and secretive nature.  
2. **Professor Archibald Crane** – A historian obsessed with rare manuscripts.  
3. **Simon the Butler** – Loyal, but with a mysterious past.  
4. **Vivian Rose** – A renowned art dealer with a penchant for rare artifacts.

**My Approach:**  
- Examine the clues for hidden meanings.  
- Interview each suspect, noting inconsistencies.  
- Reconstruct the timeline of the gala.  
- Search for overlooked evidence.

**First Steps:**

In [18]:
'''
1. First response → rich story
The model creates its own scenario:

Blackwood Manor
Missing manuscript
Defined suspects

Second prompt → completely new scene

“You arrive at the crime scene…”

Now suddenly:

New location: apartment
No manuscript
No suspects

> That means → memory is lost
Later prompts → confusion
Model says:
 “what object?”
 “missing context”

> Because:
It never saw previous outputs
We didn’t pass them back'''

'''
Summary of what happened:
first answer looks “smart” > Because our first prompt: “Imagine you are a detective…”

This is open-ended, so model: invents everything, drives the narrative

But later prompts:
depend on previous context
which is missing
'''

'\n1. First response → rich story\nThe model creates its own scenario:\n\nBlackwood Manor\nMissing manuscript\nDefined suspects\n\nSecond prompt → completely new scene\n\n“You arrive at the crime scene…”\n\nNow suddenly:\n\nNew location: apartment\nNo manuscript\nNo suspects\n\n> That means → memory is lost\n\nLater prompts → confusion\n\nModel says:\n\n“what object?”\n“missing context”\n\n> Because:\n\nIt never saw previous outputs\nYou didn’t pass them back'

In [19]:
#OPTION 3 — Maintain conversation history
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=512
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [23]:
messages = [
    {"role": "system", "content": "You are a detective solving a mystery step by step."}
]

prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

Prompt: Start the investigation.
Response: **Case File #001: The Mysterious Midnight Theft**

**Setting:**  
It’s a stormy night in the quiet town of Willowbrook. At precisely midnight, an alarm rings out from the city’s prized Willowbrook Museum. The main exhibit—an ancient jeweled necklace known as the “Emerald Serpent”—has vanished from its glass case.

**Initial Report:**  
Officer Jane Carter, the first on scene, finds:  
- A shattered display case  
- Wet footprints leading from the exhibit hall toward a side exit  
- No sign of forced entry at doors or windows  
- Security guard, Lou, unconscious but unharmed near the security office  
- Three people inside:  
    1. The museum’s director, Margaret Ellis  
    2. Visiting historian, Dr. Victor Lane  
    3. Security guard, Lou (now awake but groggy)

**Your Role:**  
You, Detective, arrive at the scene to begin your investigation.

---

**What would you like to do first?**  
1. Interview the suspects (Margaret, Victor, Lou)  
2.

In [24]:
'''
Our setup produced:

--Sequential reasoning across steps
--Evidence → inference → conclusion flow
--Consistent story memory
--Logical suspect identification

Output:
Pen found
Initials identified
Guest log matched
Suspect inferred

>> That is implicit chain-of-thought reasoning

What We did NOT strictly achieve > explicit CoT format, like:

“Step 1: …”
“Reasoning: …”
“Therefore: …”

Instead, the model:
--Embedded reasoning naturally inside narration
--Mixed story + logic
This is called:
>> Narrative reasoning, not strict CoT prompting
'''

'\nOur setup produced:\n\n--Sequential reasoning across steps\n--Evidence → inference → conclusion flow\n--Consistent story memory\n--Logical suspect identification\n\nOutput:\nPen found\nInitials identified\nGuest log matched\nSuspect inferred\n\n>> That is implicit chain-of-thought reasoning\n\nWhat We did NOT strictly achieve > explicit CoT format, like:\n\n“Step 1: …”\n“Reasoning: …”\n“Therefore: …”\n\nInstead, the model:\n--Embedded reasoning naturally inside narration\n--Mixed story + logic\nThis is called:\n>> Narrative reasoning, not strict CoT prompting\n'

In [25]:
#Improved System prompt (strict COT)
messages = [
    {
        "role": "system",
        "content": """You are a detective AI.

For every step:
- Do NOT ask the user questions
- Continue the story automatically
- Show reasoning explicitly in this format:

Observation:
Reasoning:
Conclusion:

Keep the story consistent across steps."""
    }
]

In [27]:
prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
#Evidence → reasoning → conclusion (consistently across turns)

Prompt: Start the investigation.
Response:
Observation:  
With Michael identified as the primary suspect, officers begin gathering background information. They learn Michael recently lost his job at the local factory and had an argument with his uncle, Albert Hawthorne, over money. Neighbors report seeing Michael near the mansion earlier that evening, wearing muddy boots.

Reasoning:  
Michael’s recent financial troubles and strained relationship with his uncle provide a motive. The neighbor’s sighting and the muddy boots match the evidence found at the scene. These details strengthen the case against him and suggest the break-in was premeditated, possibly to acquire something valuable or to confront Albert.

Conclusion:  
The investigation officially opens with Michael as the main suspect. Officers prepare to interview Albert Hawthorne and canvass the area for Michael’s current whereabouts, while forensic specialists analyze the key, note, and fingerprints from the scene.

Prompt: You

In [59]:
#Paramerizing
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=800
        )

        choice = response.choices[0]

        # Normal case
        if choice.message and choice.message.content:
            return choice.message.content.strip()

        # Edge case (no content)
        return f"[No content | finish_reason={choice.finish_reason}]"

    except Exception as e:
        return f"Error: {str(e)}"

def safe_get_completion(messages):
    for _ in range(3):
        res = get_completion(messages)

        if res and \
           not res.startswith("[") and \
           not res.startswith("Error") and \
           not res.startswith("Fallback") and \
           len(res.strip()) > 20:
            return res

    return "Fallback: Model failed to generate meaningful response."

In [60]:
crime_type = input("Enter crime type: ")
location = input("Enter location: ")
object_hint = input("Enter a clue/object (optional): ")

object_hint = object_hint if object_hint else "a suspicious object"

Enter crime type:  stealing
Enter location:  new hotel
Enter a clue/object (optional):  footprints at window


In [61]:
messages = [
    {
        "role": "system",
        "content": f"""
You are a detective AI solving a {crime_type} case at {location}.
You MUST ALWAYS respond with meaningful content.
If uncertain, invent reasonable details.

Format:
Observation:
Reasoning:
Conclusion:

Never return empty output.
Never skip a response.
"""
    }
]

In [64]:
prompts = [
    f"Start the investigation of a {crime_type} at {location}. Describe the victim, surroundings, and key initial clues.",

    f"You arrive at the crime scene in {location}. Describe what you observe in detail including the body, environment, and any immediate evidence.",

    f"Describe a strange object such as {object_hint} found at the scene.",

    "Explain how this object relates to the crime.",

    "Identify the most likely suspect based on all evidence."
]

In [65]:
#With memory handling
for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = safe_get_completion(messages)
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
    
    # Only store valid responses
    if not response.startswith("Fallback") and not response.startswith("["):
        messages.append({"role": "assistant", "content": response})
    else:
        print("Skipping bad response, not adding to history")

Prompt: Start the investigation of a stealing at new hotel. Describe the victim, surroundings, and key initial clues.
Response:
Observation:  
The case takes place at the recently opened "Azure Crest Hotel," a sleek high-rise with polished marble floors and modern décor. The victim is Mr. Alan Grant, a middle-aged conference attendee in his early 50s, staying in Room 512 on the fifth floor. He’s found in the lounge area of his room, visibly upset, reporting the theft of his expensive laptop and a leather briefcase containing important documents.

The room is well-kept with large windows overlooking the city, a neatly made bed, and a compact workspace. There are two coffee cups on the desk—one with lipstick stains, suggesting he had a visitor. The sliding balcony door is slightly open, and a cold breeze drifts in. There’s no sign of forced entry at the main door.

Key initial clues include:
- The open balcony door with faint muddy smudges on the floor just inside the threshold.
- Two di

In [37]:
#Another example
prompt = """
let's analyze the sentiment of the review step by step

1. Identify the Positive aspect of the review and give a score from 10
2. Identify the Negative aspect of the review and give a score from 10
3. Weight the positive and negative aspect to determine the overall sentiment.
4. provide the final sentiment classification with justification for scores used in above steps.

Review: "The product is very well designed product"
"""

In [38]:
response = get_completion(prompt)

In [ ]:
print(response)

In [ ]:
response

In [41]:
#Another example
prompt = """

let's sort the values of the list step by step

1. Start with the unsorted list.
2. Compare each elements and find the smallest value.
3. Place the samllest value in the first position.
4. Repeat the process for all the remaming elements.
5. provide the sorted list.

Sort the list : [3,1,4,6,5,9,2]

"""
response = get_completion(prompt)

In [ ]:
response

In [43]:
prompt = """

"You have 12 identical-looking balls, but one is either heavier or lighter.
You have a balance scale and can only use it three times.
Explain step-by-step how you can find the odd ball and determine whether it is heavier or lighter.
Think through the problem carefully and explain your reasoning in detail before giving the final answer."

"""
response = get_completion(prompt)

In [ ]:
response

In [45]:
#Another example
prompt = """
Let's consider which is heavier: 1000 feathers or a 30-pound weight.
I'll think through this in a few different ways and then decide which answer seems most consistent.

1. First line of reasoning: A single feather is very light, almost weightless.
So, 1000 feathers might still be quite light, possibly lighter than a 30-pound weight.

2. Second line of reasoning: 1000 is a large number, and when you add up the weight of so many feathers,
it could be quite heavy. Maybe it's heavier than a 30-pound weight.

3. Third line of reasoning: The average weight of a feather is very small. Even 1000 feathers would not add up to 30 pounds.

Considering these reasonings, the most consistent answer is: & the reason to choose the answer is :
"""

In [46]:
response = get_completion(prompt)

In [ ]:
response

In [48]:
#To work with Langchain, Install langchain related dependencies

In [49]:
#If not done already
#!pip install --upgrade "langchain>=0.3.29" "langchain-core>=1.0.0" langchain-openai

In [50]:
#If using langchain & AzureOpenAI
#from langchain_openai import AzureOpenAI
# Initialize client once
#client_lc = AzureOpenAI(
#    api_key=os.getenv("API_KEY"),
#    api_version=os.getenv("AZURE_API_VERSION"),
#    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
#    deployment_name="gpt-4o-mini", # chat completions would work with this model
#    temperature=0.5,
#    top_p=0.8,
#    max_tokens=512
#)

In [51]:
from langchain_openai import AzureChatOpenAI
# Initialize client once
client_lc = AzureChatOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1",
    temperature=0.5,
    max_tokens=512
)

In [52]:
def get_completion_lc(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

In [53]:
#Generate multiple lines of reasoning to answer the question.
#Ask the AI to evaluate these reasonings and determine the most consistent answer.

In [ ]:
prompt = """
Let's consider which is heavier: 1000 feathers or a 30-pound weight.
I'll think through this in a few different ways and then decide which answer seems most consistent.

1. First line of reasoning: A single feather is very light, almost weightless.
So, 1000 feathers might still be quite light, possibly lighter than a 30-pound weight.

2. Second line of reasoning: 1000 is a large number, and when you add up the weight of so many feathers,
it could be quite heavy. Maybe it's heavier than a 30-pound weight.

3. Third line of reasoning: The average weight of a feather is very small. Even 1000 feathers would not add up to 30 pounds.

Considering these reasonings, the most consistent answer is:
"""

response = get_completion(prompt)
print(response)

In [ ]:
prompt = """

A farmer has 17 sheeps, all but 9 run away. How many are left?
1. All but 9 ran away --> 9 are left
2. "All but 9" means 9 stayed --> 9 are left
3. Subtracting 17 - 9 --> 8 are left

Considering these reasonings, the most consistent answer is:
"""

response = get_completion(prompt)
print(response)

In [ ]:
prompt = """
I will solve the following math problem in several different ways and check if I arrive at the same answer each time.
Problem: There were 15 apples and you took away 4, how many apples do you have?

1. First approach: 15 apples - 4 apples = 11 apples, which is incorrect
2. Second approach: If i take away 4 apples, then i have 4 apples with me, which is correct
3. Third approach: Taking away 4 apples means i have 4 apples, which is correct
4. Fourth approach : Subtracting 4 from 15 will give me 11 apples, which is incorrect.

Let's see which approach gives the most consistent result.
"""
response = get_completion(prompt)
print(response)

In [57]:
#Provide a series of prompts that guide the model through a tree of thought.
#Call __get_completion__ to get a response from the AI model.
#Print both the prompt and the AI-generated response.

In [ ]:
prompt = """
Solve the problem: A farmer has 100 meters of fencing and wants to enclose the maximum area for his rectangular field. What should the dimensions be?

Let's think about this in a few ways:
1. If the field is a square, each side would be 100 / 4 = 25 meters. The area would be 25 * 25 = 625 square meters.
2. What if the field is not a square? Let's try a 4:1 ratio. The lengths would be 40 and 10 meters. The area would be 40 * 10 = 400 square meters.
3. Are there any other ratios that might give a larger area than a square or a 4:1 rectangle?

Considering these options and reason out on own and then output the best dimensions for the maximum area :
"""
response = get_completion(prompt)
print("AI Response:")
print(response)

In [ ]:
prompt = """
Let's analyze the legal case by considering multiple precedents and possible outcomes.

1. Precedent 1: A similar case where the plaintiff won.
    - Branch A: The court found that the defendant was negligent.
        - Sub-branch A1: The plaintiff was awarded damages due to clear evidence of negligence.
        - Sub-branch A2: The court ruled in favor of the plaintiff due to the defendant's breach of duty.

2. Precedent 2: A similar case where the defendant won.
    - Branch B: The court found no negligence on the defendant's part.
        - Sub-branch B1: The plaintiff failed to provide sufficient evidence.
        - Sub-branch B2: The court ruled that the plaintiff assumed the risk.

3. Precedent 3: A case with a mixed outcome.
    - Branch C: The court found both parties partially at fault.
        - Sub-branch C1: Damages were reduced based on the plaintiff's contributory negligence.
        - Sub-branch C2: The court ruled that both parties shared liability, resulting in a split decision.

4. Based on the facts of the current case, consider the most likely outcome.
"""

response = get_completion(prompt)
print(response)


In [ ]:
prompt = """
Let's plan a weekend trip by considering multiple options.

1. Option 1: Go to the mountains.
    - Branch A: If the weather is good in the mountains.
        - Sub-branch A1: You can go hiking.
        - Sub-branch A2: You can visit a nearby lake.
    - Branch B: If the weather is bad in the mountains.
        - Sub-branch B1: You will stay in a cabin and relax.
        - Sub-branch B2: You can explore local museums.

2. Option 2: Go to the beach.
    - Branch C: If the weather is sunny at the beach.
        - Sub-branch C1: You can swim in the ocean.
        - Sub-branch C2: You can sunbathe and play beach volleyball.
    - Branch D: If the weather is cloudy or rainy at the beach.
        - Sub-branch D1: You will visit indoor attractions like an aquarium.
        - Sub-branch D2: You can go shopping in beachside stores.

3. Considering all these factors, decide the best option for your weekend trip.

What should i do for my trip to Goa in month of august

"""

response = get_completion(prompt)
print(response)
